<div align="right"><a href="L05-Frequenzgang-und-Wobbelmessung.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/S05-Frequency-response-and-sweep-measurement.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S04 Analysis of a 2nd Order System](./S04-2nd-Order-System.ipynb) | [Exercise 05](./E05-Frequency-response-and-sweep-measurement.ipynb) | [Contents / Overview](./00-Python-Notebooks-Overview-EN.ipynb) | Next: [S06 Sine-Generator](./S06-Sine-Generator.ipynb)

</div>

---


### Exercise: Frequency response and sweep measurement

We will examine a system with the following Z-Transformation function:

$$
H(z) = 1 - z^{-m}
$$

**a)** Calculate the frequency-, phase-, and impulse response of the system.  
(pencil, paper)  
(really do it — you might have to solve a similar task in your oral examination anyway)

**b)** Calculate the amplitude- and phase response using the Python function `freqz()` from `scipy.signal` and plot the results (linear scale) for a range of  
$0 \leq \Omega < 2 \pi$.  
Set $m = 1$.  
Validate your calculations from part a) of this exercise.

**c)** Calculate the impulse response using the function `impz()` (provided in `helper_functions.py`) and plot the results.  
Create a zero-pole plot using the function `zplane()` (also provided in `helper_functions.py`).  
Explain what kind of filter you have.

**d)** In practice, a frequency response can usually be obtained using so-called sweep measurements.  
Using sweep measurements, a system is excited with oscillations of different frequencies.  
An LTI system responds to an oscillation with the same frequency but a different phase and amplitude.  
If the frequency is varied across the part of the spectrum that we are interested in ($0 \dots \pi$),  
the amplitude and phase response can be determined.

In this exercise, the amplitude response shall be measured using an excitation signal

$$
x(k) = \cos(\Omega_i k)
$$

Compare the results with the outcome of b).

**e)** Excite the system with a step sequence $\varepsilon(k)$ and demonstrate that the first difference of the step response equals the impulse response.  
Hint: Use the function `np.diff()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import impz, zplane
# BEGIN SOLUTION
# Filter / parameters
H = np.array([1, -1], dtype=float)   # FIR: H(z) = 1 - z^{-1}
A = np.array([1.0], dtype=float)
N = 32

# #####  Part b: Amplitude and phase response  #####
w, Hf = signal.freqz(H, A, worN=256, whole=True)  # w in [0, 2π)

# Amplitude
plt.figure()
plt.plot(w/(2*np.pi), np.abs(Hf))
plt.title('Amplitude Response')
plt.xlabel(r'$\Omega/2\pi$')
plt.ylabel('Amplitude')

# Phase (note: -angle as in MATLAB definition)
plt.figure()
plt.plot(w/(2*np.pi), -np.angle(Hf)/np.pi*180)
plt.yticks([-90, -45, 0, 45, 90])
plt.title('Phase Response')
plt.xlabel(r'$\Omega/2\pi$')
plt.ylabel('Phase in Degrees')

# #####  Part c: Impulse response, pole-zero plot  #####
h = impz(H, A, 4)   
lh = len(h)

# Impulse response (stem)
plt.figure()
(markerline, stemlines, baseline) = plt.stem(np.arange(lh), h)
plt.axis([-0.1, 3.1, -1, 1])
plt.plot([-0.1, 3.1], [0, 0], 'k')
plt.title('Impulse Response')
plt.xlabel('k')
plt.ylabel('Amplitude')

# Pole-zero plot
zplane(H, A)  

# #####  Part d: Sweep measurement – cosine excitation #####
lf = len(H)
k_sweep = np.arange(0, N - 1 + 2*(lf - 1) + 1)  # include endpoint like MATLAB
n_sweep = np.arange(0, N//2)                    # 0 .. N/2-1
erg = []

for n in n_sweep:
    sweep = np.cos(2*np.pi*n/N * k_sweep)
    foo = np.convolve(H, sweep)
    # remove transients at both ends (lf-1 samples)
    core = foo[(lf-1):len(foo)-(lf-1)]
    erg.append(np.max(np.abs(core)))
erg = np.array(erg)

# Comparison: theoretical amplitude response (standard freqz: w in [0, π])
w_th, H_th = signal.freqz(H, A, worN=N, whole=False)

plt.figure()
plt.plot(2*n_sweep/N, erg, 'r+')
plt.plot(w_th/np.pi, np.abs(H_th), 'b-')
plt.title('+ Sweep Measurement, - Theoretical Amplitude Response')
plt.xlabel(r'$\Omega/\pi$')
plt.ylabel('Amplitude')

# #####  Part e: Sweep measurement – exponential (complex) excitation #####
erg = []
for n in n_sweep:
    sweep = np.exp(1j*2*np.pi*n/N * k_sweep)
    foo = np.convolve(H, sweep)
    core = foo[(lf-1):len(foo)-(lf-1)]
    erg.append(np.max(np.abs(core)))
erg = np.array(erg)

plt.figure()
plt.plot(2*n_sweep/N, erg, 'r+')
plt.plot(w_th/np.pi, np.abs(H_th), 'b-')
plt.title('+ Sweep Measurement, - Theoretical Amplitude Response')
plt.xlabel(r'$\Omega/\pi$')
plt.ylabel('Amplitude')

# #####  Part f: Step response  #####
epsilon = np.r_[np.zeros(10), np.ones(10)]
step_response = np.convolve(h, epsilon)
hdiff = np.diff(step_response)

# Step sequence
plt.figure()
plt.stem(np.arange(len(epsilon)) - 10, epsilon)
plt.title('Step Sequence')
plt.xlabel('k')
plt.ylabel(r'$\epsilon(k)$')

# Step response
plt.figure()
plt.stem(np.arange(len(step_response)) - 10, step_response)
plt.plot([-10, 15], [0, 0], 'k')
plt.title('Step Response')
plt.xlabel('k')
plt.ylabel('Amplitude')

# Impulse response (from diff of step response)
plt.figure()
plt.stem(np.arange(len(hdiff)) - 10, hdiff)
plt.plot([-10, 15], [0, 0], 'k')
plt.title('Impulse Response')
plt.xlabel('k')
plt.ylabel('Amplitude')

plt.show()
# END SOLUTION

### Solution: frequency response and sweep measurement

**a)**

$$
H(z) = 1 - z^{-m}
$$

The **amplitude response** $\left|H(e^{j \Omega})\right|$ of
the system is calculated as:

$$
H(e^{j \Omega}) = 1 - e^{-j \Omega m}
$$

$$
\begin{aligned}
\left|H(e^{j \Omega})\right|
  &= \left| 1 - e^{-j \Omega m} \right| \\
  &= \left|  e^{-j \frac{\Omega}{2} m} \left( e^{j \frac{\Omega}{2} m} - e^{-j  \frac{\Omega}{2} m} \right) \right| \\
  &= \underbrace{\left|  e^{-j \frac{\Omega}{2} m} \right|}_{1} \left|\left( e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m} \right) \right| \\
  &= \left| \left( e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m} \right) \right| \\
  &= \left| 2 j \sin \left( \frac{\Omega}{2} m\right) \right| \\
\left|H(e^{j \Omega})\right|  &= 2 \left| \sin \left( \frac{\Omega}{2} m\right) \right|
\end{aligned}
$$

The **phase response** $b(\Omega)$ is according to the
definition:  
(Attention: The phase response is defined here with a negative sign, so that the group delay is obtained through positive derivation of the phase response  
$\tau_g(\Omega) = \frac{\mathrm{d}\, b(\Omega)}{\mathrm{d}\, \Omega}$.  
Normally, the phase response is defined positive, so that for the determination of the group delay a negative sign is needed.)

$$
H(e^{j \Omega}) = \left| H(e^{j \Omega}) \right| e^{- j b(\Omega) }
$$

is established through:

$$
\begin{aligned}
b(\Omega) &= - \arg\!\left(H(e^{j\Omega})\right) \\
          &= - \arctan\!\left(\frac{\Im\{H(e^{j\Omega})\}}{\Re\{H(e^{j\Omega})\}}\right)
\end{aligned}
$$

For this purpose, it is practical to convert the transfer function analogous to the above calculation:

$$
\begin{aligned}
H(e^{j \Omega}) &= 1 - e^{-j \Omega m} \\
                &= e^{-j \frac{\Omega}{2} m}  \left[ e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m}\right] \\
        &= \left[ \cos \left( \frac{\Omega}{2} m \right) - j \sin \left( \frac{\Omega}{2} m \right) \right]
        \left[ 2 j \sin \left( \frac{\Omega}{2} m \right) \right] \\
        &= 2 \sin^2 \left( \frac{\Omega}{2} m \right) + j\, 2 \sin \left( \frac{\Omega}{2} m \right)
        \cos \left( \frac{\Omega}{2} m \right)
\end{aligned}
$$

$$
\begin{aligned}
b(\Omega) &= - \arctan \frac{\Im\{H(e^{j\Omega})\}}{\Re\{H(e^{j\Omega})\}} \\
          &= - \arctan \frac{ 2 \sin \left( \tfrac{\Omega}{2} m \right)
        \cos \left( \tfrac{\Omega}{2} m \right) }{ 2 \sin^2 \left( \tfrac{\Omega}{2} m \right)  } \\
          &= - \arctan \frac{ \cos \left( \tfrac{\Omega}{2} m \right) }{ \sin \left( \tfrac{\Omega}{2} m \right)  } \\
          &= - \arctan \cot \left( \tfrac{\Omega}{2} m \right)  \\
          & \downarrow \quad \text{with } \arctan x = \tfrac{\pi}{2} - \operatorname{arccot} x \\
      &= - \tfrac{\pi}{2} + \operatorname{arccot}\, \cot \left( \tfrac{\Omega}{2} m \right)  \\
b(\Omega) &= - \tfrac{\pi}{2} + \tfrac{\Omega}{2} m
\end{aligned}
$$

**b)**

If the Python function `scipy.signal.freqz()` is used, the amplitude and the phase response are plotted according to the following figures:

<img src="./pngs/ldsv2a18_b1.png" width="250">  
<img src="./pngs/ldsv2a18_b2.png" width="250">  

It shows that the curve trend matches the analytical consideration in exercise part a).

**c)**

The impulse response, which is built using the helper function `impz()` (provided in `helper_functions.py`), is described in the following plots.  
Its trend can be derived directly from the Z-transfer function:

$$
h(k) = 1 \delta_0(k) - 1 \delta_0(k-m)
$$

<img src="./pngs/ldsv2a18_c1.png" width="250">  
<img src="./pngs/ldsv2a18_c2.png" width="250">  

The pole in the origin of the pole-zero plot arises because

$$
H(z) = 1 - z^{-m} = \frac{z^m - 1}{z^m}
$$

This represents a **high-pass filter**, since low frequencies are suppressed by the zeros at the position $z_0 = 1$.

**d)**

The amplitude response, which is obtained with the help of sweep measurements, corresponds with the theoretical determination.

<img src="./pngs/ldsv2a18_d.png" width="250">  

**e)**

The following figure shows the step response of the system.  
Besides the actual response, the transient response, which arises from the
finite length of the step sequence $\epsilon(k)$, can be observed.

<img src="./pngs/ldsv2a18_e1.png" width="250">  
<img src="./pngs/ldsv2a18_e2.png" width="250">  

By forming the first difference (using `np.diff()`), the impulse response of the system can be observed again.

<img src="./pngs/ldsv2a18_e3.png" width="250">  